# Kaggle-Titanic

## 导入数据

In [1]:
import numpy as np
import pandas as pd


train_data = pd.read_csv(r'data/train.csv')
test_data = pd.read_csv(r'data/test.csv')

full_data = [train_data, test_data]

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

## 数据探索与预处理的步骤与方法

### 特征工程

提取Name列中的称谓

In [ ]:
import re


def get_title(name):
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)
    return ""


for df in full_data:
    df['Title'] = df['Name'].apply(get_title)
    df['Title'] = df['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

### 补充缺失值

其中，登船港口使用众数，票价使用中位数，而年龄使用对应称谓组的中位数

In [ ]:
for df in full_data:
    df["Embarked"] = df["Embarked"].fillna(train_data["Embarked"].mode()[0])
    df["Fare"] = df['Fare'].fillna(train_data['Fare'].median())
    df['Age'] = df['Age'].fillna(df.groupby(
        'Title')['Age'].transform('median'))

### 编码

对 'Title','Sex','Embarked' 列进行onehot

In [ ]:
X_classes = ['Pclass', 'Title', 'Sex', 'Age',
             'SibSp', 'Parch', 'Fare', 'Embarked']
need_onehot = (1, 2, 7)

X_train_uncodded = train_data[X_classes].to_numpy()
X_test_uncodded = test_data[X_classes].to_numpy()
X_all_uncodded = np.concat([X_train_uncodded, X_test_uncodded])
y_train = train_data['Survived'].to_numpy()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
encoder.fit(X_all_uncodded[:, need_onehot])

one_hotted = encoder.transform(X_train_uncodded[:, need_onehot]).toarray()
X_train = np.concatenate(
    (np.delete(X_train_uncodded, need_onehot, axis=1), one_hotted), axis=1)

one_hotted = encoder.transform(X_test_uncodded[:, need_onehot]).toarray()
X_test = np.concatenate(
    (np.delete(X_test_uncodded, need_onehot, axis=1), one_hotted), axis=1)

## 各个模型的训练过程与性能比较

### 模型训练

In [ ]:
from sklearn.model_selection import GridSearchCV


def train(X, y, model, param_grid):
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X, y)

    print(grid_search.best_params_)
    print(grid_search.best_score_)

    return grid_search.best_estimator_

#### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
param_grid_rf = {
    'n_estimators': [100, 150, 200, 300, 500],
    'max_depth': range(3, 20, 1)
}
best_model_rf = train(X_train, y_train, rf, param_grid_rf)

Fitting 5 folds for each of 85 candidates, totalling 425 fits
{'max_depth': 5, 'n_estimators': 150}
0.8316364321134895


#### LogisticRegression

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
param_grid_lr = {
    'C': [1e-2, 1e-1, 1, 1e1, 1e2, 1e3],
    'penalty': ['l1', 'l2']
}
best_model_lr = train(X_train_scaled, y_train, lr,
                      param_grid_lr, )

Fitting 5 folds for each of 12 candidates, totalling 60 fits
{'C': 1, 'penalty': 'l2'}
0.8293955181721172


/home/x1879/miniconda3/envs/cs231n/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/x1879/miniconda3/envs/cs231n/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/x1879/miniconda3/envs/cs231n/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and w

#### KNeighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
param_grid_knn = {
    'n_neighbors': range(1, 16, 2)
}
best_model_knn = train(X_train_scaled, y_train, knn,
                       param_grid_knn, )

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'n_neighbors': 9}
0.8170359676103194


### 最终提交

In [ ]:
def generate_submission(model, name, X):
    y_pred = model.predict(X)
    submission = pd.DataFrame({
        'PassengerId': test_data['PassengerId'],
        'Survived': y_pred
    })
    submission.to_csv(f'sub_{name}.csv', index=False)
    return y_pred

In [ ]:
models = [best_model_rf, best_model_lr, best_model_knn]
names = ['rf', 'lr', 'knn']
Xs = [X_test, X_test_scaled, X_test_scaled]
ys = []

for model, name, X in zip(models, names, Xs):
    ys.append(generate_submission(model, name, X))

### 综合三模型投票

In [ ]:
y_all = np.where(ys[0]+ys[1]+ys[2] > 1.5, 1, 0)
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': y_all
})
submission.to_csv(f'sub_all.csv', index=False)

### 性能比较

| 模型 | 训练速度 | 准确性 | 
| :---: | :---: | :---: | 
| RandomForest | 慢 | 0.77990 | 
| LogisticRegression | 较快 | 0.77033| 
| KNeighborsClassifier | 无训练过程 | 0.77033 | 
| 三模型投票 | - | 0.78229 |

## 超参数调优的过程与最终模型的选择理由

### 超参数调优的过程

- 使用 GridSearchCV 自动调参

### 最终模型的选择理由

- 使用三模型投票，准确率最高